# Neuron data structure generation

This notebook builds a single per-neuron table for the tumor cohort. **Each row is one neuron.**

Columns:
- `subject_id` - anonymized subject (from `subj_list`)
- `insertion_index`, `unit_id` - identifiers (kept for review / later joining)
- `region` - from `region_list`
- `flair` - 1 if `opercular_list == 0`, else 0
- `grade` - from `grade_list`
- `path` - from `path_list`
- `depth` - depth from pial surface (NaN if the insertion used deep montage and therefore has no depth data)
- `waveform_cluster` - combined (waveform + spiking metrics) K-means cluster id, reproduced from the pickle (NaN if the insertion used deep montage and therefore has no depth data)
- `beh_tstat_prod` - production-activity t-statistic (NaN if the session has no speech production)
- `beh_tstat_rec` - reception-activity t-statistic (NaN if the session has no speech reception)
- `information_capacity` - per-neuron information capacity
- `spike_times` - raw full-recording spike times (seconds)

In [10]:
import numpy as np
import pandas as pd
import pickle
import fnmatch
import re
from pathlib import Path

import pynapple as nap
from scipy import stats
from scipy.stats import kstest, ttest_rel
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

# Canonical tumor-cohort session list (22 NWB files, 28 insertions).
# Every descriptive list below is indexed by INSERTION (0..27).
nwb_paths = [
    Path("/data_store2/neuropixels/nwb/old/NP93_B1/NP93_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP95_B1/NP95_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP101_B3/NP101_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP105_B1/NP105_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP113_B1/NP113_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP114_B1/NP114_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP116_B2/NP116_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP122_B1/NP122_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP128_B1/NP128_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP129_B1/NP129_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP132_B2/NP132_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP132_B3/NP132_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP136_B1/NP136_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP137_B1/NP137_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP138_B1/NP138_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B1/NP139_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B2/NP139_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP147_B2/NP147_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP150_B1/NP150_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP153_B1/NP153_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP171_B1/NP171_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP174_B3/NP174_B3.nwb"),
]

# Per-insertion descriptive info (length 28).
subj_list      = [1, 2, 3, 5, 6, 6, 6, 7, 8, 9, 10, 10, 11, 12, 13, 14, 15, 16, 16, 17, 17, 18, 19, 20, 20, 21, 22, 23]
path_list      = ['ast', 'ast', 'gbm', 'oli', 'gbm', 'gbm', 'gbm', 'gbm', 'gbm', 'ast', 'ast', 'ast', 'ast', 'oli', 'oli', 'gbm', 'ast', 'ast', 'ast', 'ast', 'ast', 'ast', 'gbm', 'ast', 'ast', 'oli', 'gbm', 'gbm']
grade_list     = [4, 4, 4, 3, 4, 4, 4, 4, 4, 3, 2, 2, 4, 2, 2, 4, 2, 2, 2, 2, 2, 2, 4, 2, 2, 3, 4, 4]
opercular_list = [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]
region_list    = ['aSTG', 'SFG', 'aSTG', 'SFG', 'vPrCG', 'vPrCG', 'vPrCG', 'pSTG', 'aMTG', 'MFG', 'aSTG', 'parsOp', 'MFG', 'PoCG', 'PoCG', 'pSTG', 'parsOr', 'vPrCG', 'pSTG', 'parsTr', 'pSTG', 'parsTr', 'SMG', 'parsTr', 'pSTG', 'vPrCG', 'aSTG', 'vPrCG']

manual_exclude_lists = [
    [2],                                                                                                                                                                              # NP93_B1.imec0
    [],                                                                                                                                                                               # NP95_B1.imec0
    [29, 45, 55],                                                                                                                                                                     # NP101_B3.imec0
    [130, 150, 151, 245, 371, 375, 377, 380, 386, 452],                                                                                                                              # NP105_B1.imec0
    [149, 156, 172, 173, 176, 217, 221, 253, 272, 276, 336, 487],                                                                                                                    # NP113_B1.imec0
    [22, 112, 149, 155, 161, 167, 176, 188, 190, 196, 213, 216, 241, 252, 255, 258, 286, 289, 307, 325, 326, 370, 395, 442, 443, 448, 449, 451, 455, 510, 549, 582, 692, 710, 717, 719, 722, 726, 729],  # NP113_B1.imec1
    [385, 448],                                                                                                                                                                       # NP113_B1.imec2
    [32, 34, 37, 39, 67, 142, 145, 175, 188, 310, 325, 331, 391, 392],                                                                                                               # NP114_B1.imec0
    [11, 15, 30, 36],                                                                                                                                                                 # NP116_B2.imec0
    [340, 382],                                                                                                                                                                       # NP122_B1.imec0
    [54, 109, 152],                                                                                                                                                                   # NP128_B1.imec0
    [],                                                                                                                                                                               # NP128_B1.imec1
    [0, 6, 13, 33, 51, 52],                                                                                                                                                           # NP129_B1.imec0
    [19, 62, 111, 151, 171, 192, 199, 200, 205, 210, 227, 229, 244, 290, 298, 311],                                                                                                  # NP132_B2.imec0
    [0, 10, 11, 15, 29, 45, 61, 112, 127, 149, 151, 158, 159, 172, 197, 209, 212, 218, 297, 315],                                                                                    # NP132_B3.imec0
    [334, 335, 333],                                                                                                                                                                  # NP136_B1.imec0
    [361],                                                                                                                                                                            # NP137_B1.imec0
    [],                                                                                                                                                                               # NP138_B1.imec0
    [169, 195, 198],                                                                                                                                                                  # NP138_B1.imec1
    [137, 149, 196, 197, 199, 206, 212, 220, 223, 231],                                                                                                                              # NP139_B1.imec0
    [129, 248, 249],                                                                                                                                                                  # NP139_B1.imec1
    [28, 59, 60, 62, 63, 82, 104, 112, 146],                                                                                                                                          # NP139_B2.imec0
    [47, 57, 59, 73, 82, 88, 116, 124, 125, 126, 127, 129, 148, 175, 194, 196, 239, 240, 242, 243, 244, 246, 253, 267, 245, 247, 248, 249, 250, 251, 254],                          # NP147_B2.imec0
    [],                                                                                                                                                                               # NP150_B1.imec0
    [34, 64, 119, 135],                                                                                                                                                               # NP150_B1.imec1
    [20, 22, 47, 71, 80, 118, 125, 134, 141, 145, 146, 166, 173, 176, 191, 270, 298, 307, 385, 414, 391, 472, 473, 490, 491],                                                        # NP153_B1.imec0
    [16, 67, 74, 78, 91, 99, 221, 224, 225, 288],                                                                                                                                     # NP171_B1.imec0
    [9, 11, 13, 23, 40, 43, 47, 59, 67, 72, 87, 104, 109, 122, 166, 169, 172, 175, 197, 199, 203, 205, 211, 212, 219, 227, 229, 230, 235, 237, 240, 251, 256, 295, 296],            # NP174_B3.imec0
]

assert len(subj_list) == len(path_list) == len(grade_list) == len(opercular_list) == len(region_list) == len(manual_exclude_lists) == 28
print("NWB files:", len(nwb_paths))
print("insertions:", len(subj_list))

NWB files: 22
insertions: 28


In [11]:
# Depth + waveform cluster from revision_waveform_tumor_v3.pkl.
# The pickle stores depth (depthAll) but NOT a cluster id, so the cluster is reproduced
# here EXACTLY as the COMBINED (waveform + spiking metrics) clustering in
# R_F2_SF4_EI_proportions_tumor.ipynb (this is the clustering that yields k=3), using only
# the data already stored in the pickle. Only the clustering analysis is reproduced; none of
# that notebook's visualization is needed to get the cluster ids.

with open("revision_waveform_tumor_v3.pkl", "rb") as f:
    wf = pickle.load(f)

indicesAll         = wf["indicesAll"]          # per-insertion arrays of good unit ids (26 insertions; PoCG excluded)
depthAll           = wf["depthAll"]            # per-insertion depths, aligned with indicesAll
waveformAll        = wf["waveformAll"]         # per-insertion peak-channel waveforms, aligned with indicesAll
firingRatesAll     = wf["firingRatesAll"]      # per-insertion firing rates, aligned with indicesAll
burstingMetricsAll = wf["burstingMetricsAll"]  # per-insertion bursting-metric dicts, aligned with indicesAll
spikeTimesAll      = wf["spikeTimesAll"]        # per-insertion full-recording spike times, aligned with indicesAll
pkl_lengths = [len(x) for x in indicesAll]
print("pkl insertions:", len(indicesAll), "| total neurons:", sum(pkl_lengths))

# Per-neuron metrics, flattened in (pkl insertion, neuron) order.
# Waveform metrics (verbatim from R_F2_SF4): width, amplitude, asymmetry, rise time, decay time.
# Firing-rate metrics (verbatim from R_F2_SF4): firing rate, burst index, ISI CV, ISI violation rate,
# spike-frequency adaptation. Sampling rate assumed 30 kHz (samples / 30 -> ms), same as the reference.
comprehensive_waveform_metrics = []
comprehensive_firing_rate_metrics = []
for ins in range(len(waveformAll)):
    for n in range(len(waveformAll[ins])):
        w = np.asarray(waveformAll[ins][n])

        peak_idx = np.argmax(w)
        trough_idx = np.argmin(w)
        peak_amp = np.max(w)
        trough_amp = np.min(w)

        spike_width = abs(peak_idx - trough_idx) / 30.0

        spike_amplitude = peak_amp - trough_amp
        if abs(trough_amp) > abs(peak_amp):
            spike_amplitude = -abs(spike_amplitude)

        spike_asymmetry = np.inf if abs(trough_amp) < 1e-10 else abs(peak_amp / trough_amp)

        baseline_end = len(w) // 10
        baseline = np.mean(w[:baseline_end])
        rising_phase = w[:peak_idx]
        rise_crossings = np.where(np.diff(np.sign(rising_phase - baseline)) > 0)[0]
        spike_rise_time = (peak_idx - rise_crossings[-1]) / 30.0 if len(rise_crossings) > 0 else peak_idx / 30.0

        baseline_start = int(len(w) * 0.9)
        baseline_d = np.mean(w[baseline_start:])
        decay_phase = w[peak_idx:]
        decay_crossings = np.where(np.diff(np.sign(decay_phase - baseline_d)) < 0)[0]
        spike_decay_time = decay_crossings[0] / 30.0 if len(decay_crossings) > 0 else (len(w) - peak_idx) / 30.0

        comprehensive_waveform_metrics.append(
            [spike_width, spike_amplitude, spike_asymmetry, spike_rise_time, spike_decay_time]
        )

        firing_rate = firingRatesAll[ins][n]
        burst_index = burstingMetricsAll[ins][n]["burst_index"]
        s = np.asarray(spikeTimesAll[ins][n])

        # ISI coefficient of variation
        if len(s) < 2:
            isi_cv = 0
        else:
            isis = np.diff(s)
            mean_isi = np.mean(isis)
            isi_cv = 0 if (len(isis) == 0 or mean_isi == 0) else np.std(isis) / mean_isi

        # ISI violation rate (2 ms refractory)
        if len(s) < 2:
            isi_violation_rate = 0
        else:
            isi_violation_rate = np.sum(np.diff(s) < 0.002) / len(s)

        # Spike-frequency adaptation (1 s windows)
        if len(s) < 10:
            spike_frequency_adaptation = 0
        else:
            total_time = s[-1] - s[0]
            n_windows = int(total_time / 1.0)
            if n_windows < 2:
                spike_frequency_adaptation = 0
            else:
                window_rates = []
                for win in range(n_windows):
                    win_start = s[0] + win * 1.0
                    win_end = win_start + 1.0
                    window_rates.append(np.sum((s >= win_start) & (s < win_end)) / 1.0)
                if len(window_rates) < 2:
                    spike_frequency_adaptation = 0
                else:
                    early_rate = np.mean(window_rates[: len(window_rates) // 2])
                    late_rate = np.mean(window_rates[len(window_rates) // 2 :])
                    spike_frequency_adaptation = 0 if early_rate == 0 else (early_rate - late_rate) / early_rate

        comprehensive_firing_rate_metrics.append(
            [firing_rate, burst_index, isi_cv, isi_violation_rate, spike_frequency_adaptation]
        )

comprehensive_waveform_array = np.array(comprehensive_waveform_metrics)
comprehensive_firing_rate_array = np.array(comprehensive_firing_rate_metrics)

# Standardize each block separately, then concatenate (verbatim from R_F2_SF4).
comprehensive_waveform_scaled = StandardScaler().fit_transform(comprehensive_waveform_array)
comprehensive_firing_rate_scaled = StandardScaler().fit_transform(comprehensive_firing_rate_array)
comprehensive_combined_data = np.column_stack([comprehensive_waveform_scaled, comprehensive_firing_rate_scaled])

# UMAP -> optimal-k (silhouette over 2..7) -> final K-means (verbatim params; yields k=3).
combined_comprehensive_umap = umap.UMAP(
    n_neighbors=15, min_dist=0.1, n_components=2, random_state=42, metric="euclidean"
).fit_transform(comprehensive_combined_data)

best_k, best_score = None, -np.inf
for k in range(2, 8):
    lab = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(combined_comprehensive_umap)
    score = silhouette_score(combined_comprehensive_umap, lab)
    print(f"k={k}: silhouette = {score:.3f}")
    if score > best_score:
        best_k, best_score = k, score
print("optimal k:", best_k)

combined_comp_labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(combined_comprehensive_umap)

# Build a join lookup keyed by the sorted unit-id signature of each pkl insertion
# (verified unique across insertions): signature -> {unit_id: (depth, cluster)}.
pkl_depth_cluster_by_signature = {}
pos = 0
for ins in range(len(indicesAll)):
    uids = np.asarray(indicesAll[ins])
    signature = tuple(int(x) for x in np.sort(uids))
    assert signature not in pkl_depth_cluster_by_signature, "duplicate pkl insertion signature"
    per_unit = {}
    for j in range(len(uids)):
        per_unit[int(uids[j])] = (float(depthAll[ins][j]), int(combined_comp_labels[pos + j]))
    pkl_depth_cluster_by_signature[signature] = per_unit
    pos += len(uids)
assert pos == len(combined_comp_labels)
print("pkl insertions indexed for join:", len(pkl_depth_cluster_by_signature))

pkl insertions: 26 | total neurons: 993


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


k=2: silhouette = 0.526
k=3: silhouette = 0.608
k=4: silhouette = 0.565
k=5: silhouette = 0.515
k=6: silhouette = 0.503
k=7: silhouette = 0.505
optimal k: 3
pkl insertions indexed for join: 26


In [12]:
# Main loop: rebuild good neurons per insertion and assemble one row per neuron.
# Filtering, production/reception t-stats, and information capacity are reproduced
# exactly from R_F1_SF1_unitbehavior.ipynb and R_F3_SF5_assemblyinfo.ipynb.

def imec_key_sorter(key):
    m = re.search(r"imec(\d+)", key)
    return int(m.group(1)) if m else float("inf")

MIN_BEHAVIOR_MINUTES = 0  # matches the reference notebooks (always uses the TaskTimes window)
TIMESCALE = 25 / 1000      # 25 ms bins for information capacity

rows = []
insertion = 0
matched_signatures = set()

for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = data.keys()

    keys = [k for k in keys if fnmatch.fnmatch(k, "*imec*")]
    ks_keys = [k for k in keys if "KS4" in k]
    if ks_keys:
        keys = ks_keys
    th8_keys = [k for k in keys if "Th=8" in k]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [k for k in keys if "Th=" in k]
        if th_keys:
            keys = th_keys
    keys = [k for k in keys if not fnmatch.fnmatch(k, "*sentgen*") and not fnmatch.fnmatch(k, "*_auto*")]
    if ("NP137" in str(nwb_paths[i])) or ("NP139_B2" in str(nwb_paths[i])):
        keys = [k for k in keys if "imec1" not in k]
    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        spike_times = data[keys[s]]
        firingRates_all = spike_times.metadata["rate"]

        if "TaskTimes" in data.keys():
            task_times = data["TaskTimes"]
        else:
            task_times = data["task_times"]
        beh_epochs = nap.IntervalSet(start=task_times.start, end=task_times.end)

        starts = np.asarray(task_times.start, dtype=float).ravel()
        ends = np.asarray(task_times.end, dtype=float).ravel()
        beh_sec = float(np.sum(ends - starts)) if starts.size == ends.size else np.nan
        beh_min = beh_sec / 60.0 if np.isfinite(beh_sec) else np.nan
        use_full_recording = (not np.isfinite(beh_min)) or (beh_min < MIN_BEHAVIOR_MINUTES)

        if use_full_recording:
            spike_times_beh = spike_times
            firingRates_beh = firingRates_all
            tmin, tmax = np.inf, -np.inf
            for u in range(len(spike_times)):
                idx = spike_times[u].as_series().index.values
                if len(idx):
                    tmin = min(tmin, float(np.min(idx)))
                    tmax = max(tmax, float(np.max(idx)))
            if np.isfinite(tmin) and np.isfinite(tmax):
                min_time, max_time = tmin, tmax
            else:
                min_time, max_time = np.nan, np.nan
        else:
            spike_times_beh = spike_times.restrict(beh_epochs)
            firingRates_beh = spike_times_beh.metadata["rate"]
            min_time = starts[0]
            max_time = ends[-1]

        # KS statistic vs uniform over [min_time, max_time]
        ks_stats = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            t = spike_times[u].as_series().index.values
            if len(t) > 1:
                ks_stats[u] = kstest((t - min_time) / (max_time - min_time), "uniform").statistic
            else:
                ks_stats[u] = np.nan

        # ISI refractory violation percentage (3 ms)
        violationPct = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            unit = spike_times[u].as_series().index
            if len(unit) < 100:
                violationPct[u] = 1
            else:
                isi = unit.diff()[1:]
                violationPct[u] = np.array(np.where(isi < 3 / 1000)).size / len(isi)

        if "KSLabel" in spike_times.metadata:
            KSLabels = spike_times.metadata["KSLabel"]
        else:
            KSLabels = spike_times.metadata["quality"]

        firingRates = firingRates_beh
        mask = (violationPct < 3 / 100) & (firingRates > 0.5) & (KSLabels != "noise") & (ks_stats < 0.3)
        indicesFinal = firingRates.index[mask]
        indicesFinal = np.setdiff1d(indicesFinal, manual_exclude_lists[insertion])

        # ---- per-insertion descriptive info ----
        subj = subj_list[insertion]
        region = region_list[insertion]
        grade = grade_list[insertion]
        path = path_list[insertion]
        flair = 1 if opercular_list[insertion] == 0 else 0

        print(f"insertion {insertion:2d}  {nwb_paths[i].stem:12s} {keys[s]:38s}  subj {subj:2d}  {region:7s}  neurons {len(indicesFinal)}")

        # ---- depth + waveform cluster from the pickle (join by unit-id signature) ----
        depth_map = {int(uid): np.nan for uid in indicesFinal}
        cluster_map = {int(uid): np.nan for uid in indicesFinal}
        signature = tuple(int(x) for x in np.sort(np.asarray(indicesFinal)))
        if signature in pkl_depth_cluster_by_signature:
            matched_signatures.add(signature)
            per_unit = pkl_depth_cluster_by_signature[signature]
            for uid in indicesFinal:
                depth_map[int(uid)] = per_unit[int(uid)][0]
                cluster_map[int(uid)] = per_unit[int(uid)][1]
        elif region != "PoCG":
            print(f"   WARNING: no pickle match for insertion {insertion} ({region}); depth/cluster set to NaN")

        # ---- production / reception t-stats (R_F1_SF1_unitbehavior) ----
        spike_times_good_beh = spike_times_beh[indicesFinal]
        prod_tstat_map = {int(uid): np.nan for uid in indicesFinal}
        rec_tstat_map = {int(uid): np.nan for uid in indicesFinal}

        if "ProdSpeechWords" in data:
            sp = data["ProdSpeechWords"]
            prod_starts = nap.Ts(sp.start).as_series().index + (-0.10)
            prod_ends = nap.Ts(sp.end).as_series().index + 0.0
            for uid in indicesFinal:
                spikes = spike_times_good_beh[uid]
                threshold_rate = spike_times_good_beh[uid].rate
                rates = np.zeros(len(prod_starts))
                for u in range(len(prod_starts)):
                    iv = nap.IntervalSet(start=prod_starts[u], end=prod_ends[u])
                    rates[u] = len(spikes.restrict(iv)) / (prod_ends[u] - prod_starts[u])
                if len(rates) > 1:
                    tval, _ = ttest_rel(rates, np.full_like(rates, threshold_rate), alternative="greater")
                else:
                    tval = np.nan
                prod_tstat_map[int(uid)] = float(tval)

        if ("StimSpeechWords" in data) or ("mfa_stim_words" in data):
            sp = data["StimSpeechWords"] if "StimSpeechWords" in data else data["mfa_stim_words"]
            sens_starts = nap.Ts(sp.start).as_series().index + 0.0
            sens_ends = nap.Ts(sp.end).as_series().index + 0.10
            for uid in indicesFinal:
                spikes = spike_times_good_beh[uid]
                threshold_rate = spike_times_good_beh[uid].rate
                rates = np.zeros(len(sens_starts))
                for u in range(len(sens_starts)):
                    iv = nap.IntervalSet(start=sens_starts[u], end=sens_ends[u])
                    rates[u] = len(spikes.restrict(iv)) / (sens_ends[u] - sens_starts[u])
                if len(rates) > 1:
                    tval, _ = ttest_rel(rates, np.full_like(rates, threshold_rate), alternative="greater")
                else:
                    tval = np.nan
                rec_tstat_map[int(uid)] = float(tval)

        # ---- per-neuron information capacity (R_F3_SF5_assemblyinfo) ----
        # NOTE: unlike the reference, this is computed for EVERY neuron regardless of whether the
        # insertion has detected assemblies (n_pcs). This is a reference dataset, so we always want
        # a per-neuron capacity value. The per-neuron `capacity` math is otherwise identical.
        info_cap_map = {int(uid): np.nan for uid in indicesFinal}
        if len(indicesFinal) > 0:
            spike_times_good_full = spike_times[indicesFinal]   # full recording, matches reference
            spikeCountMatrix = spike_times_good_full.count(bin_size=TIMESCALE).values
            firingRateMatrix = spikeCountMatrix / TIMESCALE
            firingRateMatrix = stats.zscore(firingRateMatrix, axis=0)
            if firingRateMatrix.shape[0] > 1 and firingRateMatrix.shape[1] > 0:
                for j, uid in enumerate(indicesFinal):
                    neuron_activity = firingRateMatrix[:, j]
                    valid = neuron_activity[~np.isnan(neuron_activity)]
                    if len(valid) < 10:
                        capacity = np.nan
                    else:
                        n_bins = min(20, len(valid) // 5)
                        if n_bins < 5:
                            n_bins = 5
                        hist, _ = np.histogram(valid, bins=n_bins)
                        prob = hist / np.sum(hist)
                        prob = prob[prob > 0]
                        entropy_val = -np.sum(prob * np.log2(prob)) if len(prob) >= 2 else np.nan

                        x = valid[:-1]
                        y = valid[1:]
                        mi_bins = min(10, len(valid) // 10)
                        vm = ~(np.isnan(x) | np.isnan(y))
                        if np.sum(vm) < 2:
                            mi_temporal = np.nan
                        else:
                            hist_2d, _, _ = np.histogram2d(x[vm], y[vm], bins=mi_bins)
                            total = np.sum(hist_2d)
                            p_xy = hist_2d / total
                            p_x = np.sum(hist_2d, axis=1) / total
                            p_y = np.sum(hist_2d, axis=0) / total
                            mi_temporal = 0.0
                            for a in range(len(p_x)):
                                for b in range(len(p_y)):
                                    if p_xy[a, b] > 0 and p_x[a] > 0 and p_y[b] > 0:
                                        mi_temporal += p_xy[a, b] * np.log2(p_xy[a, b] / (p_x[a] * p_y[b]))

                        if not np.isnan(entropy_val) and not np.isnan(mi_temporal):
                            capacity = entropy_val - mi_temporal
                        elif not np.isnan(entropy_val):
                            capacity = entropy_val
                        else:
                            capacity = np.nan
                    info_cap_map[int(uid)] = capacity

        # ---- assemble one row per neuron ----
        for uid in indicesFinal:
            spike_times_unit = np.asarray(spike_times[uid].as_series().index.values, dtype=float)
            rows.append({
                "subject_id": int(subj),
                "insertion_index": int(insertion),
                "unit_id": int(uid),
                "region": region,
                "flair": int(flair),
                "grade": int(grade),
                "path": path,
                "depth": depth_map[int(uid)],
                "waveform_cluster": cluster_map[int(uid)],
                "beh_tstat_prod": prod_tstat_map[int(uid)],
                "beh_tstat_rec": rec_tstat_map[int(uid)],
                "information_capacity": info_cap_map[int(uid)],
                "spike_times": spike_times_unit,
            })

        insertion += 1

assert insertion == len(subj_list), (insertion, len(subj_list))
print("\\ninsertions processed:", insertion)
print("pkl insertions matched:", len(matched_signatures), "/", len(pkl_depth_cluster_by_signature))
print("total neurons:", len(rows))

/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/base_class.py:60: UserWarning: Some epochs have no duration
  self.time_support = IntervalSet(start=self.index[0], end=self.index[-1])
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/base_class.py:62: RuntimeWarning: divide by zero encountered in scalar divide
  self.rate = self.index.shape[0] / np.sum(
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/py

insertion  0  NP93_B1      NP93_B1_g0_imec0_withoutKSmc            subj  1  aSTG     neurons 1


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(


insertion  1  NP95_B1      NP95_B1_g0_imec1                        subj  2  SFG      neurons 57


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)


...

insertion 26  NP171_B1     catgt_NP171_B1_g0_imec0_KS4_Th=8        subj 22  aSTG     neurons 17


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(


insertion 27  NP174_B3     catgt_NP174_B3_g0_imec0_KS4_Th=8        subj 23  vPrCG    neurons 24


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)


\ninsertions processed: 28
pkl insertions matched: 26 / 26
total neurons: 1196


In [13]:
# Assemble the final per-neuron table and save it.
column_order = [
    "subject_id", "insertion_index", "unit_id",
    "region", "flair", "grade", "path",
    "depth", "waveform_cluster",
    "beh_tstat_prod", "beh_tstat_rec", "information_capacity",
    "spike_times",
]
neuron_data = pd.DataFrame(rows)[column_order]

print("shape:", neuron_data.shape)
print()
print("coverage:")
print("  depth present:               ", int(neuron_data["depth"].notna().sum()))
print("  waveform_cluster present:    ", int(neuron_data["waveform_cluster"].notna().sum()))
print("  beh_tstat_prod present:      ", int(neuron_data["beh_tstat_prod"].notna().sum()))
print("  beh_tstat_rec present:       ", int(neuron_data["beh_tstat_rec"].notna().sum()))
print("  information_capacity present:", int(neuron_data["information_capacity"].notna().sum()))
print()
print("waveform_cluster distribution:")
print(neuron_data["waveform_cluster"].value_counts(dropna=False).sort_index())
print()
neuron_data.drop(columns=["spike_times"]).head(20)

shape: (1196, 13)

coverage:
  depth present:                993
  waveform_cluster present:     993
  beh_tstat_prod present:       1162
  beh_tstat_rec present:        910
  information_capacity present: 1196

waveform_cluster distribution:
waveform_cluster
0.0    355
1.0    246
2.0    392
NaN    203
Name: count, dtype: int64



,subject_id,insertion_index,unit_id,region,flair,grade,path,depth,waveform_cluster,beh_tstat_prod,beh_tstat_rec,information_capacity
0,1,0,57,aSTG,0,4,ast,1037.045644,2.0,0.701725,1.884418,0.227711
1,2,1,2,SFG,1,4,ast,2706.205578,2.0,-0.088638,2.865468,0.384714
2,2,1,5,SFG,1,4,ast,2488.409692,0.0,1.176335,1.119614,0.340189
3,2,1,6,SFG,1,4,ast,6103.892565,1.0,0.393468,1.279904,0.300103
4,2,1,7,SFG,1,4,ast,6037.103305,1.0,1.641370,-7.582167,0.473722
5,2,1,9,SFG,1,4,ast,1990.067077,1.0,-1.723172,-0.375249,0.856180
6,2,1,10,SFG,1,4,ast,5811.061094,2.0,1.010002,3.484346,0.316239
7,2,1,11,SFG,1,4,ast,5758.813108,1.0,-1.870688,2.440585,0.633449
8,2,1,12,SFG,1,4,ast,1746.549459,2.0,1.555408,-1.534500,0.254698
9,2,1,14,SFG,1,4,ast,5489.586930,1.0,-3.249226,2.427108,0.248451


In [14]:
with open("neuron_data_structure.pkl", "wb") as f:
    pickle.dump(neuron_data, f, protocol=pickle.HIGHEST_PROTOCOL)
print("saved neuron_data_structure.pkl  (rows:", len(neuron_data), ")")

saved neuron_data_structure.pkl  (rows: 1196 )
